In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Atenuarea erorilor prin rețele tensoriale (TEM): O Funcție Qiskit de la Algorithmiq
> **Note:** Funcțiile Qiskit sunt o funcționalitate experimentală disponibilă doar utilizatorilor din planul IBM Quantum&reg; Premium, Flex și On-Prem (prin API-ul IBM Quantum Platform). Acestea se află în stadiu de previzualizare și pot suferi modificări.


<details>
<summary><b>Versiuni pachete</b></summary>

Codul de pe această pagină a fost dezvoltat folosind cerințele de mai jos.
Recomandăm folosirea acestor versiuni sau a unora mai noi.

```
qiskit[all]~=2.3.0
qiskit-ibm-catalog~=0.11.0
```
</details>
## Prezentare generală
Metoda TEM (Tensor-network Error Mitigation) de la Algorithmiq este un algoritm
hibrid cuantic-clasic conceput pentru atenuarea zgomotului exclusiv în etapa de
post-procesare clasică. Cu TEM, poți calcula valorile de așteptare ale
observabililor, atenuând erorile inevitabile induse de zgomot ce apar pe
hardware-ul cuantic, cu precizie sporită și eficiență a costurilor, ceea ce îl
face o opțiune extrem de atractivă atât pentru cercetătorii în domeniul cuantic,
cât și pentru practicienii din industrie.

Metoda constă în construirea unei rețele tensoriale care reprezintă inversul
canalului de zgomot global ce afectează starea procesorului cuantic și apoi în
aplicarea transformării asupra rezultatelor măsurătorilor complet informaționale
obținute din starea zgomotoasă, pentru a obține estimatori nebiasați ai
observabililor.

Ca avantaj, TEM valorifică măsurătorile complet informaționale pentru a oferi
acces la un set vast de valori de așteptare atenuate ale observabililor și are un
overhead de eșantionare optim pe hardware-ul cuantic, conform descrierii din
Filippov et al. (2023), [arXiv:2307.11740](https://arxiv.org/abs/2307.11740), și
Filippov et al. (2024), [arXiv:2403.13542](https://arxiv.org/abs/2403.13542).
Overhead-ul de măsurare se referă la numărul de măsurători suplimentare necesare
pentru atenuarea eficientă a erorilor, un factor critic în fezabilitatea
calculelor cuantice. Prin urmare, TEM are potențialul de a permite avantajul
cuantic în scenarii complexe, cum ar fi aplicații în domeniile haosului cuantic,
fizicii multi-corpuri, dinamicii Hubbard și simulărilor de chimie a moleculelor mici.

Principalele caracteristici și beneficii ale TEM pot fi rezumate astfel:

1.  **Overhead de măsurare optim**: TEM este optim față de
[limitele teoretice](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.131.210601),
ceea ce înseamnă că nicio metodă nu poate obține un overhead de măsurare mai
mic. Cu alte cuvinte, TEM necesită numărul minim de măsurători suplimentare
pentru atenuarea erorilor. Aceasta implică, la rândul său, că TEM utilizează
un timp de execuție cuantic minim.
2.  **Eficiență a costurilor**: Deoarece TEM gestionează atenuarea zgomotului
exclusiv în etapa de post-procesare, nu este nevoie să se adauge circuite
suplimentare calculatorului cuantic, ceea ce nu doar că face calculul mai
ieftin, ci și diminuează riscul introducerii unor erori suplimentare din
cauza imperfecțiunilor dispozitivelor cuantice.
3.  **Estimarea mai multor observabili**: Datorită măsurătorilor complet
informaționale, TEM estimează eficient mai mulți observabili cu aceleași
date de măsurare de la calculatorul cuantic.
4.  **Atenuarea erorilor de măsurare**: Funcția TEM Qiskit include și o
[metodă proprietară de atenuare a erorilor de citire](https://journals.aps.org/prresearch/abstract/10.1103/PhysRevResearch.5.033154)
capabilă să reducă semnificativ erorile de readout după o scurtă rulare de
calibrare.
5.  **Precizie**: TEM îmbunătățește semnificativ precizia și fiabilitatea
simulărilor cuantice digitale, făcând algoritmii cuantici mai exacți și mai
de încredere.
## Descriere
Funcția TEM îți permite să obții valori de așteptare atenuate ale erorilor
pentru mai mulți observabili pe un Circuit cuantic cu un overhead de eșantionare
minim. Circuitul este măsurat cu o măsură de operator cu valoare pozitivă
complet informațională (IC-POVM), iar rezultatele măsurătorilor colectate sunt
procesate pe un calculator clasic. Această măsurătoare este folosită pentru a
aplica metodele de rețele tensoriale și a construi o hartă de inversare a
zgomotului. Funcția aplică o transformare care inversează complet întregul circuit
zgomotos folosind rețele tensoriale pentru a reprezenta straturile zgomotoase.

![Schema TEM](../docs/images/guides/algorithmiq-tem/tem_scheme.svg "Estimarea atenuată a erorii unui observabil O prin post-procesarea rezultatelor de măsurare ale procesorului cuantic zgomotos. U și N denotă o operație cuantică ideală și harta de zgomot asociată, care poate fi în general nelocală (și extinsă la casetele gri). D reprezintă un tensor de operatori duali efectelor din măsurarea IC. Modulul de atenuare a zgomotului M este o rețea tensorială contractată eficient din mijloc spre exterior. Prima iterație a contracției este reprezentată de linia punctată violet, a doua de linia întreruptă, iar a treia de linia continuă.")

Odată ce circuitele sunt trimise funcției, acestea sunt transpilate și
optimizate pentru a minimiza numărul de straturi cu porți cu doi Qubiți (porțile
mai zgomotoase pe dispozitivele cuantice). Zgomotul care afectează straturile
este învățat prin
[Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/noise-learner-noise-learner)
folosind un model de zgomot sparse Pauli-Lindblad conform descrierii din E. van
den Berg, Z. Minev, A. Kandala, K. Temme, Nat. Phys. (2023).
[arXiv:2201.09866](https://arxiv.org/abs/2201.09866).

Modelul de zgomot este o descriere precisă a zgomotului de pe dispozitiv,
capabilă să capteze caracteristici subtile, inclusiv interferențele între Qubiți.
Totuși, zgomotul de pe dispozitive poate fluctua și deriva, iar zgomotul
învățat s-ar putea să nu fie precis la momentul la care se face estimarea. Acest
lucru poate duce la rezultate inexacte.
## Primii pași
Autentifică-te folosind [cheia API IBM Quantum Platform](http://quantum.cloud.ibm.com/) și selectează funcția TEM după cum urmează. (Acest fragment presupune că ți-ai [salvat deja contul](/guides/functions#install-qiskit-functions-catalog-client) în mediul local.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

tem_function_name = "algorithmiq/tem"

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Load your function
tem = catalog.load(tem_function_name)

## Exemplu
Fragmentul următor arată un exemplu în care TEM este utilizat pentru a calcula valorile de așteptare ale unui observabil dat un Circuit cuantic simplu.

In [2]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

# Create a quantum circuit
qc = QuantumCircuit(3)
qc.u(0.4, 0.9, -0.3, 0)
qc.u(-0.4, 0.2, 1.3, 1)
qc.u(-1.2, -1.2, 0.3, 2)
for _ in range(2):
    qc.barrier()
    qc.cx(0, 1)
    qc.cx(2, 1)
    qc.barrier()
    qc.u(0.4, 0.9, -0.3, 0)
    qc.u(-0.4, 0.2, 1.3, 1)
    qc.u(-1.2, -1.2, 0.3, 2)

# Define the observables
observable = SparsePauliOp("IYX", 1.0)

# Define the execution options
pub = (qc, [observable])
options = {"default_precision": 0.02}

# Define backend to use. TEM will choose the least-busy device reported by IBM if not specified
backend_name = "ibm_torino"

job = tem.run(pubs=[pub], backend_name=backend_name, options=options)

Folosește API-urile Qiskit Serverless pentru a verifica statusul sarcinii tale Qiskit Function:

In [3]:
print(job.status())

QUEUED


You can return the results as:

In [4]:
result = job.result()
evs = result[0].data.evs

Poți obține rezultatele astfel:

In [5]:
print(job.result())

PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(1,), dtype=float64>), stds=np.ndarray(<shape=(1,), dtype=float64>)), metadata={'evs_non_mitigated': array([-0.06314623]), 'stds_non_mitigated': array([0.05917147]), 'evs_mitigated_no_readout_mitigation': array([-0.06411205]), 'stds_mitigated_no_readout_mitigation': array([0.05992467]), 'evs_non_mitigated_with_readout_mitigation': array([-0.07028881]), 'stds_non_mitigated_with_readout_mitigation': array([0.06353934])})], metadata={'resource_usage': {'RUNNING: OPTIMIZING_FOR_HARDWARE': {'CPU_TIME': 0.915754}, 'RUNNING: WAITING_FOR_QPU': {'CPU_TIME': 18.804865}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 10.433445}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 159.0}}})


> **Info:** Valoarea așteptată pentru circuitul fără zgomot pentru operatorul dat ar trebui să fie în jurul valorii `0.18409094298943401`.
## Intrări
**Parametri**

Nume | Tip | Descriere | Obligatoriu | Implicit | Exemplu
-- | -- | -- | -- | -- | --
`pubs` | Iterable[EstimatorPubLike] | Un iterabil de obiecte de tip PUB (primitive unified bloc), cum ar fi tupluri `(circuit, observables)` sau `(circuit, observables, parameters, precision)`. Vezi [Prezentare generală a PUB-urilor](/guides/primitive-input-output#overview-of-pubs) pentru mai multe informații. Dacă este transmis un circuit non-ISA, va fi transpilat cu setări optime. Dacă este transmis un circuit ISA, nu va fi transpilat; în acest caz, observabilul trebuie definit pe întreg QPU-ul. | Da | N/A | (circuit, observables)
`backend_name` | str | Numele Backend-ului pe care se face interogarea.| Nu | Dacă nu este furnizat, va fi utilizat Backend-ul cel mai puțin ocupat. | "ibm_fez"
`options` | dict | Opțiuni de intrare. Vezi secțiunea `Options` pentru mai multe detalii. | Nu | Vezi secțiunea `Options` pentru mai multe detalii.| {"max_bond_dimension": 100\}

> **Caution:** TEM are în prezent următoarele limitări:
> 
>   - Circuitele parametrizate nu sunt acceptate. Argumentul parameters trebuie setat la `None` dacă precizia este specificată. Această restricție va fi eliminată în versiunile viitoare.
>   - Sunt acceptate doar circuite fără bucle. Această restricție va fi eliminată în versiunile viitoare.
>   - Porțile non-unitare, cum ar fi reset, measure și toate formele de control flow, nu sunt acceptate. Suportul pentru reset va fi adăugat în versiunile următoare.
### Opțiuni
Un dicționar care conține opțiunile avansate pentru TEM. Dicționarul poate conține cheile din tabelul următor. Dacă oricare dintre opțiuni nu este furnizată, va fi folosită valoarea implicită listată în tabel. Valorile implicite sunt potrivite pentru utilizarea tipică a TEM.

Nume | Alegeri | Descriere  | Implicit
-- | -- | -- | --
`tem_max_bond_dimension` | int | Dimensiunea maximă de legătură care va fi utilizată pentru rețelele tensoriale. | 500 |
`tem_compression_cutoff` | float | Valoarea de prag utilizată pentru rețelele tensoriale. | 1e-16
`compute_shadows_bias_from_observable` | bool | Un indicator boolean care specifică dacă distorsiunea pentru protocolul de măsurare classical shadows ar trebui adaptată observabilului PUB sau nu. Dacă este False, va fi utilizat protocolul classical shadows (probabilitate egală de a măsura Z, X, Y).| False
`shadows_bias` | np.ndarray | Distorsiunea utilizată pentru protocolul randomizat de măsurare classical shadows, un array 1d sau 2d de dimensiune 3 sau formă (num_qubits, 3) respectiv. Ordinea este ZXY | np.array([1 / 3, 1 / 3, 1 / 3])
`max_execution_time` | int sau `None` | Timpul maxim de execuție pe QPU în secunde. Dacă timpul de execuție depășește această valoare, sarcina va fi anulată. Dacă este `None`, va fi aplicată o limită implicită setată de Qiskit Runtime. | `None`
`num_randomizations` | int | Numărul de randomizări care vor fi utilizate pentru învățarea zgomotului și twirling-ul porților. | 32
`max_layers_to_learn` | int | Numărul maxim de straturi unice de învățat. | 4
`mitigate_readout_error` | bool | Un indicator Boolean care specifică dacă să se efectueze sau nu atenuarea erorilor de readout. | True
`num_readout_calibration_shots` | int | Numărul de shot-uri utilizate pentru atenuarea erorilor de readout. | 10000
`default_precision` | float | Precizia implicită utilizată pentru PUB-urile pentru care precizia nu este specificată. |0.02
`seed` | int sau `None` | Setează sămânța generatorului de numere aleatoare pentru reproductibilitate. Dacă este `None`, sămânța nu este setată. | None
## Ieșiri
Un Qiskit [PrimitiveResults](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) care conține rezultatul atenuat TEM. Rezultatul pentru fiecare PUB este returnat ca un [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) care conține următoarele câmpuri:

Nume |Tip | Descriere
-- | -- | --
data | DataBin | Un Qiskit [DataBin](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin) care conține observabilul atenuat TEM și eroarea sa standard. DataBin are următoarele câmpuri: <ul><li>`evs`: Valoarea observabilului atenuat TEM.</li><li>`stds`: Eroarea standard a observabilului atenuat TEM.</li></ul>
metadata | dict | Un dicționar care conține rezultate suplimentare. Dicționarul conține următoarele chei: <ul><li>`"evs_non_mitigated"`: Valoarea observabilului fără atenuarea erorilor.</li><li>`"stds_non_mitigated"`: Eroarea standard a rezultatului fără atenuarea erorilor.</li><li>`"evs_mitigated_no_readout_mitigation"`: Valoarea observabilului cu atenuarea erorilor, dar fără atenuarea erorilor de readout.</li><li>`"stds_mitigated_no_readout_mitigation"`: Eroarea standard a rezultatului cu atenuarea erorilor, dar fără atenuarea erorilor de readout.</li><li>`"evs_non_mitigated_with_readout_mitigation"`: Valoarea observabilului fără atenuarea erorilor, dar cu atenuarea erorilor de readout.</li><li>`"stds_non_mitigated_with_readout_mitigation"`: Eroarea standard a rezultatului fără atenuarea erorilor, dar cu atenuarea erorilor de readout.</li></ul>
## Obținerea mesajelor de eroare
Dacă statusul sarcinii tale este ERROR, folosește job.result() pentru a obține mesajul de eroare astfel: